<a href="https://colab.research.google.com/github/HinnaNayab/DeepLearning/blob/Applied-AI/Tutorial_8B_Object_Detection_Faster_RCNN_(Custom_Model).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!free -h

import psutil

ram = psutil.virtual_memory()
print("Total RAM:", round(ram.total / (1024**3), 2), "GB")
print("Available RAM:", round(ram.available / (1024**3), 2), "GB")

               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.2Gi       7.4Gi        14Mi       3.4Gi        10Gi
Swap:             0B          0B          0B
Total RAM: 12.67 GB
Available RAM: 10.5 GB


In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G  107M  113G   1% /


In [ ]:
import zipfile, json, os, glob, shutil, cv2
import numpy as np


ZIP_PATH = "/content/polarbearsdata.zip"
JSON_PATH = "/content/labels-COCO-JSON.json"

# 1. Unzipping
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall("/content/raw_images")

# 2. Finding all images regardless of subfolder names
all_imgs = glob.glob("/content/raw_images/**/*.*", recursive=True)
img_map = {os.path.basename(p): p for p in all_imgs if p.lower().endswith(('.jpg', '.jpeg', '.png'))}
print(f"Found {len(img_map)} images in zip.")

base_out = "/content/"
for split in ["train", "val"]:
    os.makedirs(f"{base_out}/{split}/images", exist_ok=True)
    os.makedirs(f"{base_out}/{split}/labels", exist_ok=True)
    os.makedirs(f"{base_out}/{split}/masks", exist_ok=True)

with open(JSON_PATH) as f:
    coco = json.load(f)

id_to_img = {img["id"]: img for img in coco["images"]}
img_to_anns = {img_id: [] for img_id in id_to_img}
for ann in coco["annotations"]:
    img_to_anns[ann["image_id"]].append(ann)

labeled_ids = [i for i, anns in img_to_anns.items() if len(anns) > 0 and id_to_img[i]["file_name"] in img_map]
split_idx = int(0.8 * len(labeled_ids))

for idx, img_id in enumerate(labeled_ids):
    split = "train" if idx < split_idx else "val"
    info = id_to_img[img_id]
    fname = info["file_name"]
    h, w = info["height"], info["width"]
    base_name = os.path.splitext(fname)[0]

    shutil.copy(img_map[fname], f"{base_out}/{split}/images/{fname}")
    mask = np.zeros((h, w), dtype=np.uint8)
    yolo_lines = []

    for ann in img_to_anns[img_id]:
        bx, by, bw, bh = ann["bbox"]
        yolo_lines.append(f"0 {(bx + bw/2)/w:.6f} {(by + bh/2)/h:.6f} {bw/w:.6f} {bh/h:.6f}")

        if "segmentation" in ann and len(ann["segmentation"]) > 0:
            for seg in ann["segmentation"]:
                pts = np.array(seg, dtype=np.int32).reshape((-1, 2))
                cv2.fillPoly(mask, [pts], 255)
        else:
            cv2.rectangle(mask, (int(bx), int(by)), (int(bx+bw), int(by+bh)), 255, -1)

    cv2.imwrite(f"{base_out}/{split}/masks/{base_name}.png", mask)
    with open(f"{base_out}/{split}/labels/{base_name}.txt", "w") as f:
        f.write("\n".join(yolo_lines))

with open(f"{base_out}/data.yaml", "w") as f:
    f.write(f"train: {base_out}/train/images\nval: {base_out}/val/images\nc: 1\nnames: ['polarbear']\n")

print(f"SUCCESS! Prepared {len(labeled_ids)} images ({split_idx} train, {len(labeled_ids)-split_idx} val).")

Found 146 images in zip.
SUCCESS! Prepared 145 images (116 train, 29 val).


there are 145 images, even though in my folder it shows 150,because some images are the type webp, and i think i havent specified that

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [ ]:
#defining the backbone network
def build_backbone(input_shape=(224, 224, 3)):
    inputs = keras.Input(shape=input_shape)
    x = keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    x = keras.layers.MaxPooling2D((2, 2))(x)
    x = keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D((2, 2))(x)
    x = keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = keras.layers.MaxPooling2D((2, 2))(x)
    outputs = keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    return keras.Model(inputs, outputs)
backbone = build_backbone()

In [ ]:
#Constructing the Region Proposal Network (RPN)
def build_rpn(feature_map):
    rpn_conv = keras.layers.Conv2D(512, (3, 3), padding="same", activation="relu")(feature_map)
    rpn_class = keras.layers.Conv2D(9 * 2, (1, 1), activation="sigmoid")(rpn_conv) # 9 anchors, 2 classes (object/non-object)
    rpn_bbox = keras.layers.Conv2D(9 * 4, (1, 1))(rpn_conv) # 9 anchors, 4 bounding box values
    return rpn_class, rpn_bbox

In [ ]:
#Implementing the ROI pooling layer
class ROIPoolingLayer(tf.keras.layers.Layer):
    def __init__(self, pool_size, **kwargs):
        super(ROIPoolingLayer, self).__init__(**kwargs)
        self.pool_size = pool_size


    def call(self, inputs):
        feature_map, rois = inputs
        rois = tf.reshape(rois, (-1, 4))
        batch_indices = tf.zeros((tf.shape(rois)[0],), dtype=tf.int32)
        pooled_rois = tf.image.crop_and_resize(
            feature_map, boxes=rois, box_indices=batch_indices, crop_size=self.pool_size
        )
        return pooled_rois

    def compute_output_shape(self, input_shape):
        feature_map_shape, rois_shape = input_shape
        num_rois = rois_shape[0]
        return (num_rois, self.pool_size[0], self.pool_size[1], feature_map_shape[-1])

roi_pooling_layer = ROIPoolingLayer((7, 7))

In [ ]:
#Creating Classification and Regression Heads
def build_heads(pooled_rois, num_classes):
    flatten = keras.layers.Flatten()(pooled_rois)
    dense = keras.layers.Dense(1024, activation="relu")(flatten)
    classifier = keras.layers.Dense(num_classes, activation="softmax")(dense)
    bbox_regressor = keras.layers.Dense(num_classes * 4)(dense) # 4 values per bounding box
    return classifier, bbox_regressor

In [ ]:
#Assembling the Complete Faster R-CNN Model
def build_faster_rcnn(num_classes, input_shape=(224, 224, 3)):
    input_image = keras.Input(shape=input_shape)
    rois = keras.Input(shape=(None, 4)) # ROIs input
    #Backbone (feature extraction)
    backbone = build_backbone(input_shape)
    feature_map = backbone(input_image)
    #Region proposal network
    rpn_class, rpn_bbox = build_rpn(feature_map)
    # ROI Pooling
    pooled_rois = roi_pooling_layer([feature_map, rois])
    # Classification & Regression heads
    classifier, bbox_regressor = build_heads(pooled_rois, num_classes)
    # Buildingthe complete model
    model = keras.Model(inputs=[input_image, rois], outputs=[rpn_class, rpn_bbox, classifier, bbox_regressor])
    return model

In [ ]:
#Printing Summary of the model
num_classes = 80 #number of classes for COCO dataset (example)
faster_rcnn = build_faster_rcnn(num_classes)
faster_rcnn.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_4        │ (None, 28, 28,    │  1,550,976 │ input_layer_8[0]… │
│ (Functional)        │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_9       │ (None, None, 4)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ roi_pooling_layer_1 │ (None, 7, 7, 512) │          0 │ functional_4[0][… │
│ (ROIPoolingLayer)   │                   │            │ input_layer_9[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 25088)     │          0 │ roi_pooling_laye… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_26 (Conv2D)  │ (None, 28, 28,    │  2,359,808 │ functional_4[0][… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 1024)      │ 25,691,136 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_27 (Conv2D)  │ (None, 28, 28,    │      9,234 │ conv2d_26[0][0]   │
│                     │ 18)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_28 (Conv2D)  │ (None, 28, 28,    │     18,468 │ conv2d_26[0][0]   │
│                     │ 36)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 80)        │     82,000 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 320)       │    328,000 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 30,039,622 (114.59 MB)

 Trainable params: 30,039,622 (114.59 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#calculating IOU
def calculate_iou(box1, box2):
    ymin1, xmin1, ymax1, xmax1 = box1
    ymin2, xmin2, ymax2, xmax2 = box2
    inter_xmin = max(xmin1, xmin2)
    inter_ymin = max(ymin1, ymin2)
    inter_xmax = min(xmax1, xmax2)
    inter_ymax = min(ymax1, ymax2)
    inter_area = max(0, inter_xmax - inter_xmin) * max(0, inter_ymax - inter_ymin)
    area1 = (xmax1 - xmin1) * (ymax1 - ymin1)
    area2 = (xmax2 - xmin2) * (ymax2 - ymin2)
    union_area = area1 + area2 - inter_area
    iou = inter_area / union_area if union_area != 0 else 0
    return iou

#calculating Precision & Recall
def calculate_precision_recall(pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, iou_threshold=0.5):
    tp, fp = 0, 0
    matched_gt = set()
    for i, pred_box in enumerate(pred_boxes):
        if pred_scores[i] < iou_threshold:
            continue
        best_iou = 0
        best_gt_index = -1
        for j, gt_box in enumerate(gt_boxes):
            if pred_labels[i] == gt_labels[j] and j not in matched_gt:
                iou = calculate_iou(pred_box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_index = j
        if best_iou >= iou_threshold:
            print(f"Match found! Best IoU: {best_iou:.4f}")
            tp += 1
            matched_gt.add(best_gt_index)
        else:
            if best_gt_index != -1:
                print(f"No match (IoU too low): Best IoU was {best_iou:.4f}")
            else:
                print("No match found for predicted box (different class or no overlap)")
            fp += 1
    fn = len(gt_boxes) - len(matched_gt)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    print(f"Calculated Metrics -> Precision: {precision:.4f}, Recall: {recall:.4f}")
    return precision, recall

In [ ]:
#calculating Average Precision & mAP
def calculate_ap(precisions, recalls):
    precisions = [0.0] + precisions + [0.0]
    recalls = [0.0] + recalls + [1.0]
    for i in range(len(precisions) - 1, 0, -1):
        precisions[i - 1] = max(precisions[i - 1], precisions[i])
    indices = [i for i in range(1, len(recalls)) if recalls[i] != recalls[i - 1]]
    ap = sum((recalls[i] - recalls[i - 1]) * precisions[i] for i in indices)
    return ap

def calculate_map(predictions, ground_truths, num_classes, iou_threshold=0.5):
    average_precisions = []
    for class_id in range(1, num_classes):
        all_precisions = []
        all_recalls = []
        has_data = False

        for pred, gt in zip(predictions, ground_truths):
            pred_boxes = [box for i, box in enumerate(pred['boxes']) if pred['labels'][i] == class_id]
            pred_scores = [score for i, score in enumerate(pred['scores']) if pred['labels'][i] == class_id]
            gt_boxes = [box for i, box in enumerate(gt['boxes']) if gt['labels'][i] == class_id]

            # Only evaluate if there are actual predictions or ground truths for this class
            if len(pred_boxes) > 0 or len(gt_boxes) > 0:
                has_data = True
                print(f"\n--- Evaluating Class {class_id} ---")
                precision, recall = calculate_precision_recall(
                    pred_boxes, pred_scores, [class_id] * len(pred_boxes), gt_boxes, [class_id] * len(gt_boxes), iou_threshold
                )
                all_precisions.append(precision)
                all_recalls.append(recall)

        if has_data:
            ap = calculate_ap(all_precisions, all_recalls)
            average_precisions.append(ap)
            print(f"Class {class_id} AP: {ap:.4f}")

    MAP = np.mean(average_precisions) if len(average_precisions) > 0 else 0
    return MAP

#sampling values for Obtaining mAP
predictions = [
    {'boxes': [[0.1, 0.1, 0.4, 0.4], [0.5, 0.5, 0.8, 0.8]], 'scores': [0.9, 0.7], 'labels': [1, 2]}
]
ground_truths = [
    {'boxes': [[0.12, 0.12, 0.42, 0.42], [0.5, 0.5, 0.8, 0.8]], 'labels': [1, 2]}
]
num_classes = 80
mean_ap = calculate_map(predictions, ground_truths, num_classes)
print(f"\nMean Average Precision (mAP): {mean_ap:.2f}")


--- Evaluating Class 1 ---
Match found! Best IoU: 0.7717
Calculated Metrics -> Precision: 1.0000, Recall: 1.0000
Class 1 AP: 1.0000

--- Evaluating Class 2 ---
Match found! Best IoU: 1.0000
Calculated Metrics -> Precision: 1.0000, Recall: 1.0000
Class 2 AP: 1.0000

Mean Average Precision (mAP): 1.00
